In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")
RAW_DIR = Path("../data/raw")

df = pd.read_parquet(PROCESSED_DIR / "matches_clean.parquet")
df = df.sort_values("date").reset_index(drop=True)

print(f"Loaded {len(df)} matches")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")

Loaded 32287 matches
Date range: 1990-01-12 to 2026-06-10


In [2]:
def calculate_elo(df,
                  initial_rating=1500,
                  k_friendly=20,
                  k_competitive=40,
                  k_worldcup=60,
                  decay_half_life_years=10):  # matches 10yrs ago = half weight
    
    ratings = {}
    history = []
    latest_date = df["date"].max()

    def get_rating(team):
        return ratings.get(team, initial_rating)

    def expected(rating_a, rating_b):
        return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))

    def get_k(row):
        # base K by tournament type
        if row["is_worldcup"]:   base_k = k_worldcup
        elif row["is_friendly"]: base_k = k_friendly
        else:                     base_k = k_competitive

        # time decay — more recent matches matter more
        years_ago = (latest_date - row["date"]).days / 365.25
        decay = 0.5 ** (years_ago / decay_half_life_years)
        # clamp between 0.5 and 1.0 so old matches still count
        decay = max(0.5, min(1.0, decay))
        return base_k * decay

    for _, row in df.iterrows():
        home = row["home_team"]
        away = row["away_team"]
        home_elo = get_rating(home)
        away_elo = get_rating(away)
        home_exp = expected(home_elo, away_elo)
        away_exp = 1 - home_exp

        if row["outcome"] == "home_win":
            home_act, away_act = 1.0, 0.0
        elif row["outcome"] == "away_win":
            home_act, away_act = 0.0, 1.0
        else:
            home_act, away_act = 0.5, 0.5

        k = get_k(row)

        history.append({
            "date":            row["date"],
            "home_team":       home,
            "away_team":       away,
            "home_elo_before": round(home_elo, 1),
            "away_elo_before": round(away_elo, 1),
            "elo_diff":        round(home_elo - away_elo, 1),
            "home_exp_prob":   round(home_exp, 4),
            "tournament":      row["tournament"],
            "is_worldcup":     row["is_worldcup"],
            "outcome":         row["outcome"],
            "home_score":      row["home_score"],
            "away_score":      row["away_score"],
        })

        ratings[home] = home_elo + k * (home_act - home_exp)
        ratings[away] = away_elo + k * (away_act - away_exp)

    return pd.DataFrame(history), ratings


elo_df, final_ratings = calculate_elo(df)
print(f"Elo calculated for {len(elo_df)} matches")
print(f"Teams rated: {len(final_ratings)}")

Elo calculated for 32287 matches
Teams rated: 326


In [3]:
ratings_df = pd.DataFrame([
    {"team": team, "elo": round(rating, 1)}
    for team, rating in final_ratings.items()
]).sort_values("elo", ascending=False).reset_index(drop=True)

ratings_df.index += 1  # start rank from 1
print("Top 30 teams by Elo (as of June 10, 2026):")
print(ratings_df.head(30).to_string())

Top 30 teams by Elo (as of June 10, 2026):
             team     elo
1           Spain  1979.8
2       Argentina  1954.7
3          France  1918.7
4         Morocco  1896.4
5         England  1885.4
6           Japan  1880.8
7        Portugal  1871.0
8         Germany  1857.4
9          Brazil  1855.7
10       Colombia  1843.2
11    Netherlands  1842.2
12         Mexico  1840.7
13        Ecuador  1828.2
14      Australia  1809.5
15        Croatia  1809.4
16           Iran  1807.5
17        Senegal  1803.2
18    South Korea  1802.7
19         Turkey  1801.9
20        Algeria  1800.2
21          Italy  1797.0
22        Uruguay  1787.2
23        Belgium  1782.2
24  United States  1775.2
25         Norway  1774.5
26        Nigeria  1772.9
27         Canada  1770.8
28    Switzerland  1769.8
29     Uzbekistan  1760.8
30        Denmark  1758.2


In [4]:
wc2026_groups = {
    "A": ["Mexico", "South Korea", "Czech Republic", "South Africa"],
    "B": ["Canada", "Bosnia and Herzegovina", "Switzerland", "Qatar"],
    "C": ["Brazil", "Morocco", "Haiti", "Scotland"],
    "D": ["United States", "Paraguay", "Australia", "Turkey"],
    "E": ["Germany", "Curaçao", "Ivory Coast", "Ecuador"],
    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "H": ["Spain", "Cape Verde", "Saudi Arabia", "Uruguay"],
    "I": ["France", "Senegal", "Iraq", "Norway"],
    "J": ["Argentina", "Algeria", "Austria", "Jordan"],
    "K": ["Portugal", "DR Congo", "Uzbekistan", "Colombia"],
    "L": ["England", "Croatia", "Ghana", "Panama"],
}

print("WC 2026 teams — Elo ratings:")
print("-" * 50)
for group, teams in wc2026_groups.items():
    print(f"\n  Group {group}:")
    for team in teams:
        elo = final_ratings.get(team, 1500)
        rank = ratings_df[ratings_df["team"] == team].index
        rank_str = str(rank[0]) if len(rank) > 0 else "?"
        print(f"    {team:<30} Elo: {elo:>7.1f}  Rank: #{rank_str}")

WC 2026 teams — Elo ratings:
--------------------------------------------------

  Group A:
    Mexico                         Elo:  1840.7  Rank: #12
    South Korea                    Elo:  1802.7  Rank: #18
    Czech Republic                 Elo:  1677.6  Rank: #43
    South Africa                   Elo:  1638.9  Rank: #58

  Group B:
    Canada                         Elo:  1770.8  Rank: #27
    Bosnia and Herzegovina         Elo:  1566.4  Rank: #91
    Switzerland                    Elo:  1769.8  Rank: #28
    Qatar                          Elo:  1591.2  Rank: #78

  Group C:
    Brazil                         Elo:  1855.7  Rank: #9
    Morocco                        Elo:  1896.4  Rank: #4
    Haiti                          Elo:  1623.2  Rank: #66
    Scotland                       Elo:  1701.7  Rank: #40

  Group D:
    United States                  Elo:  1775.2  Rank: #24
    Paraguay                       Elo:  1740.1  Rank: #32
    Australia                      Elo:  1809.5 

In [5]:
elo_path     = PROCESSED_DIR / "elo_history.parquet"
ratings_path = PROCESSED_DIR / "elo_ratings_current.csv"

elo_df.to_parquet(elo_path, index=False)
ratings_df.to_csv(ratings_path, index=False)

print(f"Saved Elo history        → {elo_path}")
print(f"Saved current ratings    → {ratings_path}")
print(f"\nTop 5 sanity check:")
print(ratings_df.head(5).to_string())
print("\nNext: 04_feature_engineering.ipynb")

Saved Elo history        → ..\data\processed\elo_history.parquet
Saved current ratings    → ..\data\processed\elo_ratings_current.csv

Top 5 sanity check:
        team     elo
1      Spain  1979.8
2  Argentina  1954.7
3     France  1918.7
4    Morocco  1896.4
5    England  1885.4

Next: 04_feature_engineering.ipynb
